# Phase 2: Supervised Fine-Tuning (30% Subject Split)

This notebook implements the second phase of the experimental pipeline.
It operates exclusively on the 30% of subjects held out during Phase 1, ensuring
strict subject-level separation. Synthetic clinical scores (HDRS, YMRS) are assigned
to each subject to simulate the bipolar disorder classification task described by Corponi et al.

### 1. Repository Cloning
Cloning the repository before dependency installation, so the dataset loading step executes with the native NumPy version.

In [ ]:
import os, sys, shutil

WORK_DIR = '/kaggle/working/e4selflearning'
os.chdir('/kaggle/working')

if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)

print("Cloning repository...")
!git clone https://github.com/april-tools/e4selflearning.git {WORK_DIR}
print("Repository ready.")

### 2. Dataset Integration
Linking the pre-processed dataset. Executed before the NumPy downgrade to ensure correct deserialization of the metadata.pkl binary file.

In [ ]:
import os, glob, shutil, pickle

TARGET_DIR = '/kaggle/working/e4selflearning/data/preprocessed/unsegmented_unlabelled'
if os.path.exists(TARGET_DIR): shutil.rmtree(TARGET_DIR)
os.makedirs(TARGET_DIR, exist_ok=True)

print("Searching for dataset in Kaggle Input directory...")
meta_files = glob.glob('/kaggle/input/**/metadata.pkl', recursive=True)
if not meta_files:
    raise FileNotFoundError("Error: metadata.pkl not found.")
src_dir = os.path.dirname(meta_files[0])
print(f"Source path: {src_dir}")

for item in os.listdir(src_dir):
    s, d = os.path.join(src_dir, item), os.path.join(TARGET_DIR, item)
    if item == 'metadata.pkl':
        shutil.copy2(s, d)
    else:
        os.symlink(s, d)

meta_p = os.path.join(TARGET_DIR, 'metadata.pkl')
with open(meta_p, 'rb') as f: meta = pickle.load(f)

new_sessions = {}
found_count = 0
for root, dirs, files in os.walk(TARGET_DIR, followlinks=True):
    if 'channels.h5' in files:
        clean_name = os.path.basename(root)
        new_rel_id = os.path.relpath(root, TARGET_DIR).replace('\\', '/')
        for old_id in meta['sessions_info'].keys():
            if old_id.endswith(clean_name):
                new_sessions[new_rel_id] = meta['sessions_info'][old_id]
                found_count += 1
                break

if found_count > 0:
    meta['sessions_info'] = new_sessions
    with open(meta_p, 'wb') as f: pickle.dump(meta, f)
    print(f"Loading complete. Mapped sessions: {found_count}")

### 3. Dependency Installation
Installing the required dependencies with pinned versions after dataset loading.

In [ ]:
print("Installing dependencies...")
!pip uninstall -y -q numpy scipy pandas
!pip install --no-cache-dir -q "numpy==1.26.4" "scipy==1.12.0" "pandas<2.2" biopsykit flirt ruamel.yaml biosppy hrv-analysis neurokit2 distinctipy joypy peakutils lightning torchmetrics
print("Dependencies installed.")

### 4. Classifier Patch
Replacing train_ann.py with a version that supports non-numeric session identifiers and subject-level internal splitting.

In [ ]:
import os
os.chdir('/kaggle/working/e4selflearning')

PATCH = '''import argparse, json, os, pickle, typing as t, numpy as np, hashlib
from time import time
import torch
from torch.utils.data import DataLoader
from torchmetrics.functional.classification import binary_accuracy
from tqdm import tqdm
from timebase import criterions
from timebase.data.dataset import ClassificationDataset
from timebase.models.models import Classifier, Critic, get_models
from timebase.utils import utils

def robust_session_id(filename):
    dirname = os.path.basename(os.path.dirname(filename))
    try: return int(dirname)
    except: return int(hashlib.md5(dirname.encode()).hexdigest(), 16) % 10**8

def robust_segment_id(filename):
    basename = os.path.basename(filename).replace(".h5", "")
    try: return int(basename)
    except: return int(hashlib.md5(basename.encode()).hexdigest(), 16) % 10**8

ClassificationDataset.session_id = staticmethod(robust_session_id)
ClassificationDataset.segment_id = staticmethod(robust_segment_id)

def load(d, device): return {k: v.to(device) for k, v in d.items()}

def load_pre_trained_parameters(args, classifier, path2pretraining_res):
    filename = None
    for r,d,f in os.walk(path2pretraining_res):
        if "model_state.pt" in f: filename = os.path.join(r, "model_state.pt"); break
    if not filename: raise FileNotFoundError(f"model_state.pt not found in {path2pretraining_res}")
    ckpt = torch.load(filename, map_location=args.device)
    sd = classifier.sslearner.state_dict()
    sd.update({k: v for k, v in ckpt["model"].items() if any(m in k for m in ["channel_embedding", "feature_encoder"])})
    classifier.sslearner.load_state_dict(sd)

def train(args, ds, classifier, criteria, opt_classifier, epoch):
    results = {}
    classifier.to(args.device).train(True)
    for batch in tqdm(ds, desc="Train", disable=args.verbose <= 1):
        inputs = load(batch["data"], args.device)
        targets = batch["target"].to(args.device)
        out, rep = classifier(inputs)
        loss = criteria.criterion_classifier(y_true=targets, y_pred=out)
        loss.backward(); opt_classifier.step(); opt_classifier.zero_grad()
        utils.update_dict(target=results, source={"loss/classifier": loss.detach().cpu(), "metrics/acc": binary_accuracy(out.detach().cpu(), targets.detach().cpu())})
    for k, v in results.items(): results[k] = torch.mean(torch.stack(v)).item()
    return results

def main(args):
    utils.set_random_seed(args.seed)
    if not os.path.exists(args.output_dir): os.makedirs(args.output_dir)
    utils.get_device(args)

    user_dataset_path = args.dataset
    user_output_dir = args.output_dir

    yaml_path = None
    for r,d,f in os.walk(args.path2pretraining_res):
        if "args.yaml" in f: yaml_path = r; break
    if yaml_path:
        try:
            utils.load_args(args, dir=yaml_path)
            args.dataset = user_dataset_path
            args.output_dir = user_output_dir
        except: pass

    with open(os.path.join(args.dataset, "metadata.pkl"), "rb") as f: meta = pickle.load(f)
    with open(os.path.join(args.dataset, "stats.pkl"), "rb") as f: stats_dict = pickle.load(f)
    cache_key = list(stats_dict.keys())[0]
    stats = stats_dict[cache_key]
    n_subs = len(np.unique(meta["sessions_labels"]["Sub_ID"]))

    defaults = {
        "pretext_task": "masked_prediction", "representation_module": "transformer",
        "emb_dim": 128, "mlp_dim": 128, "num_blocks": 3, "num_heads": 4, "num_units": 128,
        "input_shapes": {c: [1, 512] for c in ["ACC_x","ACC_y","ACC_z","BVP","EDA","HR","TEMP","EDA_tonic","EDA_phasic","EDA_smna"]},
        "scaling_mode": 2, "use_wandb": False, "split_mode": 0, "task_mode": 1,
        "critic_score_lambda": 0.0, "num_train_subjects": n_subs
    }
    for k, v in defaults.items():
        if not hasattr(args, k): setattr(args, k, v)

    for c in args.input_shapes.keys():
        if c not in stats: stats[c] = {"min": 0, "max": 1, "mean": 0.5, "std": 1.0, "median": 0.5, "iqr": 0.2}
        elif "std" not in stats[c]: stats[c]["std"] = 1.0

    meta["sessions_paths"] = np.array(meta["sessions_paths"])
    for k in meta["sessions_labels"].keys():
        meta["sessions_labels"][k] = np.array(meta["sessions_labels"][k])

    sub_ids = meta["sessions_labels"]["Sub_ID"]
    unique_subs = np.unique(sub_ids)

    if len(unique_subs) > 1:
        np.random.shuffle(unique_subs)
        split = int(len(unique_subs) * 0.8)
        tr_subs, val_subs = unique_subs[:split], unique_subs[split:]
        tr_idx = np.where(np.isin(sub_ids, tr_subs))[0]
        val_idx = np.where(np.isin(sub_ids, val_subs))[0]
        print(f"Internal split -> Train: {len(tr_subs)} subjects, Validation: {len(val_subs)} subjects")
    else:
        n = len(sub_ids)
        indices = np.arange(n)
        np.random.shuffle(indices)
        split = int(n * 0.8)
        tr_idx, val_idx = indices[:split], indices[split:]
        print(f"Internal split (segment-level) -> Train: {len(tr_idx)}, Validation: {len(val_idx)}")

    def get_loader(idx):
        filenames = meta["sessions_paths"][idx]
        ds = ClassificationDataset(args,
                                   filenames=filenames,
                                   labels={k: v[idx] for k, v in meta["sessions_labels"].items()},
                                   rec_ids=filenames,
                                   stats=stats,
                                   recording_id_str_to_num={p: i for i, p in enumerate(meta["sessions_paths"])},
                                   subject_ids=meta["sessions_labels"]["Sub_ID"][idx])
        return DataLoader(ds, batch_size=args.batch_size, shuffle=True, num_workers=args.num_workers)

    tr_ds = get_loader(tr_idx)
    val_ds = get_loader(val_idx)

    classifier, critic = get_models(args, summary=None)
    load_pre_trained_parameters(args, classifier, args.path2pretraining_res)

    opt_classifier = torch.optim.AdamW(classifier.parameters(), lr=args.lr)
    criteria = criterions.get_criterion(args)

    for epoch in range(1, args.epochs + 1):
        res = train(args, tr_ds, classifier, criteria, opt_classifier, epoch)
        print(f"Epoch {epoch:02d} | Loss: {res['loss/classifier']:.4f} | Acc: {res['metrics/acc']:.4f}")

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--dataset", type=str, required=True)
    parser.add_argument("--output_dir", type=str, required=True)
    parser.add_argument("--path2pretraining_res", type=str, required=True)
    parser.add_argument("--epochs", type=int, default=5)
    parser.add_argument("--batch_size", type=int, default=16)
    parser.add_argument("--lr", type=float, default=0.001)
    parser.add_argument("--verbose", type=int, default=1)
    parser.add_argument("--seed", type=int, default=1234)
    parser.add_argument("--device", type=str, default=None)
    parser.add_argument("--num_workers", type=int, default=2)
    args = parser.parse_known_args()[0]
    main(args)
'''

with open("train_ann.py", "w", encoding="utf-8") as f:
    f.write(PATCH)
print("Patch applied to train_ann.py")

### 5. Neural Segmentation
Generating fixed-length windows from the continuous recordings.

In [ ]:
import os
os.chdir('/kaggle/working/e4selflearning')
BASE = 'data/preprocessed/unsegmented_unlabelled'
SEG_DIR = 'data/preprocessed/sl512_ss256_unlabelled'
print("Starting segmentation process...")
!PYTHONPATH=src python segment.py --data_dir {BASE} --output_dir {SEG_DIR} --segment_length 512 --segmentation_mode 1 --step_size 256 --num_workers 2 --overwrite
print("Segmentation complete.")

### 6. Subject-Level Split (30% Holdout) and Synthetic Labeling
Retrieving the 30% of subjects not used in Phase 1 using the same random seed (42). Synthetic HDRS and YMRS scores simulate the bipolar disorder classification scenario.

In [ ]:
import pickle, os, numpy as np, re

SEG_DIR = '/kaggle/working/e4selflearning/data/preprocessed/sl512_ss256_unlabelled'
TGT_DIR = '/kaggle/working/e4selflearning/data/preprocessed/sl512_ss256_fase2_labeled'
os.makedirs(TGT_DIR, exist_ok=True)

meta_path = os.path.join(SEG_DIR, "metadata.pkl")
with open(meta_path, "rb") as f: meta = pickle.load(f)

all_paths = meta["sessions_paths"]
if "Sub_ID" not in meta.get("sessions_labels", {}):
    meta.setdefault("sessions_labels", {})["Sub_ID"] = [np.nan] * len(all_paths)

sub_ids = []
for path in all_paths:
    p_norm = path.replace('\\', '/').replace('//', '/')
    parts = p_norm.split('/')
    final_sid = None
    for pt in parts:
        if 'sl512' in pt or 'ss256' in pt: continue
        match = re.search(r'[sS]ubject_?(\d+)|[sS](\d+)', pt)
        if match:
            val = match.group(1) or match.group(2)
            final_sid = f"{parts[0][:3].lower()}_{val}"
            break
    if not final_sid and len(parts) >= 2:
        final_sid = f"folder_{parts[-2]}"
    sub_ids.append(final_sid)

unique_subs = np.unique(sub_ids)
unique_subs.sort()
np.random.seed(42)  # Same seed as Phase 1 to obtain the complementary partition
np.random.shuffle(unique_subs)

split_idx = int(len(unique_subs) * 0.70)
fase2_subs = set(unique_subs[split_idx:])  # 30% holdout

filtered_paths, filtered_sub_ids = [], []
for p, s in zip(all_paths, sub_ids):
    if s in fase2_subs:
        filtered_paths.append(p)
        filtered_sub_ids.append(s)

unique_fase2 = np.unique(filtered_sub_ids)
name_to_id = {name: i for i, name in enumerate(unique_fase2)}
numeric_sub_ids = [name_to_id[name] for name in filtered_sub_ids]

sub_status_map = {sid: np.random.randint(0, 2) for sid in range(len(unique_fase2))}
status, hdrs, ymrs = [], [], []
for s_id in numeric_sub_ids:
    s_status = sub_status_map[s_id]
    status.append(float(s_status))
    if s_status == 0:
        hdrs.append(np.random.uniform(0, 7)); ymrs.append(np.random.uniform(0, 7))
    else:
        mood = np.random.choice(['dep', 'man', 'mix'])
        if mood == 'dep':
            hdrs.append(np.random.uniform(15, 30)); ymrs.append(np.random.uniform(0, 5))
        elif mood == 'man':
            hdrs.append(np.random.uniform(0, 5)); ymrs.append(np.random.uniform(15, 30))
        else:
            hdrs.append(np.random.uniform(12, 25)); ymrs.append(np.random.uniform(12, 25))

meta["sessions_paths"] = np.array(filtered_paths)
meta["sessions_labels"] = {
    "status": np.array(status),
    "YMRS_SUM": np.array(ymrs),
    "HDRS_SUM": np.array(hdrs),
    "time": np.zeros(len(filtered_paths), dtype=float),
    "Sub_ID": np.array(numeric_sub_ids)
}
with open(os.path.join(TGT_DIR, "metadata.pkl"), "wb") as f: pickle.dump(meta, f)

channels = ['ACC_x','ACC_y','ACC_z','BVP','EDA','HR','TEMP','EDA_tonic','EDA_phasic','EDA_smna']
base_stats = {c: {'min': 0, 'max': 1, 'mean': 0.5, 'std': 1.0, 'median': 0.5, 'iqr': 0.2} for c in channels}
stats_path = os.path.join(TGT_DIR, "stats.pkl")
with open(stats_path, "wb") as f: pickle.dump({stats_path: base_stats}, f)

print(f"Phase 2 dataset constructed.")
print(f"Phase 2 partition (30%): {len(unique_fase2)} subjects, {len(filtered_paths)} labelled segments.")

### 7. Supervised Fine-Tuning
Loading the pre-trained SSL encoder weights from Phase 1 and training the classification head on the labelled 30% holdout partition.

In [ ]:
import os, glob
os.chdir('/kaggle/working/e4selflearning')

print("Searching for pre-trained SSL weights...")
pre_res_files = glob.glob('/kaggle/input/**/model_state.pt', recursive=True)
if not pre_res_files:
    raise FileNotFoundError("Error: model_state.pt not found. Ensure the Phase 1 output has been added as an input dataset.")
PRE_RES = os.path.dirname(os.path.dirname(pre_res_files[0]))
print(f"Weights found in: {PRE_RES}")

TGT_DIR = '/kaggle/working/e4selflearning/data/preprocessed/sl512_ss256_fase2_labeled'
print("Starting supervised fine-tuning...")
!PYTHONPATH=src python train_ann.py --dataset {TGT_DIR} --path2pretraining_res {PRE_RES} --output_dir "output_finetune" --epochs 50 --batch_size 16